# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [1]:
# your code here

import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# Load the dataset and copy it
df = pd.read_csv("tic-tac-toe.csv")
backup_df = df.copy()

#Inspect the dataset
display(df.shape)
print(df.head())
print(df.info())
print(df['class'].value_counts())


(958, 10)

  TL TM TR ML MM MR BL BM BR  class
0  x  x  x  x  o  o  x  o  o   True
1  x  x  x  x  o  o  o  x  o   True
2  x  x  x  x  o  o  o  o  x   True
3  x  x  x  x  o  o  o  b  b   True
4  x  x  x  x  o  o  b  o  b   True
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 958 entries, 0 to 957
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   TL      958 non-null    object
 1   TM      958 non-null    object
 2   TR      958 non-null    object
 3   ML      958 non-null    object
 4   MM      958 non-null    object
 5   MR      958 non-null    object
 6   BL      958 non-null    object
 7   BM      958 non-null    object
 8   BR      958 non-null    object
 9   class   958 non-null    bool  
dtypes: bool(1), object(9)
memory usage: 68.4+ KB
None
class
True     626
False    332
Name: count, dtype: int64


## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [2]:
# Encode with labelEncoder
encoder = LabelEncoder()
encoded_df =df.apply(encoder.fit_transform)

# Split input and target

X = encoded_df.drop("class",axis = 1)
y = encoded_df["class"]
y_encoded = encoder.fit_transform(y)
# Normalize with MinMaxScaler (so we get better model)
scale = MinMaxScaler()

X_scaled = scale.fit_transform(X)

X_scaled[:5]


array([[1. , 1. , 1. , 1. , 0.5, 0.5, 1. , 0.5, 0.5],
       [1. , 1. , 1. , 1. , 0.5, 0.5, 0.5, 1. , 0.5],
       [1. , 1. , 1. , 1. , 0.5, 0.5, 0.5, 0.5, 1. ],
       [1. , 1. , 1. , 1. , 0.5, 0.5, 0.5, 0. , 0. ],
       [1. , 1. , 1. , 1. , 0.5, 0.5, 0. , 0.5, 0. ]])

In [3]:
# your code here
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)


In [4]:
model = Sequential()
# Input layer + first hidden layer (ReLU)
model.add(Dense(32, input_dim=9, activation="relu"))
# Second hidden layer (ReLU)
model.add(Dense(16, activation="relu"))
# Sigmoid, X wins or not
model.add(Dense(2, activation="sigmoid"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [5]:
# Model summarization
model.summary()

# Compile the model
model.compile(optimizer="adam",
loss = "sparse_categorical_crossentropy",
metrics=["accuracy"])


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 882 (3.45 KB)

 Trainable params: 882 (3.45 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# Train the model

model.fit(X_train, y_train, epochs = 50, batch_size = 64, verbose=1)

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)

print(f"The test acurracy gives us: {round(accuracy,2)}")

# Save the model
model.save("tic_tac_toe.keras")

Epoch 1/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 75ms/step - accuracy: 0.6726 - loss: 0.6405
Epoch 2/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6606 - loss: 0.6403 
Epoch 3/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6377 - loss: 0.6446 
Epoch 4/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6738 - loss: 0.6173 
Epoch 5/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6376 - loss: 0.6380 
Epoch 6/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6544 - loss: 0.6213 
Epoch 7/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6681 - loss: 0.6053 
Epoch 8/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6523 - loss: 0.6240 
Epoch 9/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6550 - loss: 0.6104 
Epoch 10/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6663 - loss: 0.6088 
Epoch 11/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6926 - loss: 0.5892 
Epoch 12/50
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 

## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [7]:
# your code here

import numpy as np
from tensorflow.keras.models import load_model
# Load the saved model
model = load_model("tic_tac_toe.keras")

sample = np.random.choice(len(X_test), size = 5, replace = False)

X_sample = X_test[sample]
y_sample = y_test.iloc[sample]

#
prob_pred = model.predict(X_sample)
classes_pred = np.argmax(prob_pred, axis = 1)

print("Predictions:", classes_pred)
print("True Labels:", list(y_sample))

#
df_results = pd.DataFrame({
    "Predicted": classes_pred,
    "Actual": y_sample.values
})

print(df_results)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 315ms/step
Predictions: [0 1 1 1 1]
True Labels: [0, 0, 1, 1, 1]
   Predicted  Actual
0          0       0
1          1       0
2          1       1
3          1       1
4          1       1


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [8]:
# your code here

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

optimizer = tf.keras.optimizers.Adam(learning_rate = 0.001)
sgd_opti = tf.keras.optimizers.SGD(learning_rate = 0.01, momentum = 0.9)
rms_opti = tf.keras.optimizers.RMSprop(learning_rate = 0.001)
adg_opti = tf.keras.optimizers.Adagrad(learning_rate = 0.01)

second_model = Sequential([tf.keras.Input(shape =(9,)),
Dense(64, activation ="relu"),Dense(32, activation ="relu"),
Dense(16, activation ="relu"),
Dropout(0.2),
Dense(2, activation = "sigmoid")
])




In [9]:
second_model.compile(
    optimizer = optimizer,
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"]
)

History = second_model.fit(
  X_train, y_train,
  epochs = 150,
  batch_size = 64,
  verbose = 1,
  validation_split = 0.2
)



Epoch 1/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 313ms/step - accuracy: 0.4025 - loss: 0.7119 - val_accuracy: 0.6883 - val_loss: 0.6789
Epoch 2/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6211 - loss: 0.6770 - val_accuracy: 0.6818 - val_loss: 0.6568
Epoch 3/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6622 - loss: 0.6535 - val_accuracy: 0.6818 - val_loss: 0.6381
Epoch 4/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6344 - loss: 0.6534 - val_accuracy: 0.6818 - val_loss: 0.6285
Epoch 5/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6588 - loss: 0.6312 - val_accuracy: 0.6818 - val_loss: 0.6202
Epoch 6/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6536 - loss: 0.6253 - val_accuracy: 0.6818 - val_loss: 0.6160
Epoch 7/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6333 - loss: 0.6273 - val_accuracy: 0.6818 - val_loss: 0.6138
Epoch 8/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6478 - loss: 0.6101 - val_accuracy: 0.6948 -

In [10]:
loss, accuracy = second_model.evaluate(X_test, y_test)
print(f"Improved Model Accuracy: {round(accuracy,4)}")
print(f"Improved Model Loss: {round(loss,4)}")

second_model.save("tic_tac_toe_V2.keras")

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8767 - loss: 0.2863  
Improved Model Accuracy: 0.875
Improved Model Loss: 0.3402


In [11]:
import numpy as np
import pandas as pd

from tensorflow.keras.models import load_model

# Load the saved model
model = load_model("tic_tac_toe_V2.keras")

# Pick a few random rows from the test set
indices = np.random.choice(len(X_test), size=5, replace=False)

X_sample = X_test[indices]
y_sample = y_test.iloc[indices]

# Make predictions (probabilities)
pred_probs = model.predict(X_sample)

# Convert probabilities → class labels
pred_classes = np.argmax(pred_probs, axis=1)

print("Predictions:", pred_classes)
print("True Labels:", list(y_sample))

# Predictions in a nice table
df_results = pd.DataFrame({
    "Predicted": pred_classes,
    "Actual": y_sample.values
})

print(df_results)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 300ms/step
Predictions: [1 0 0 1 0]
True Labels: [1, 1, 0, 0, 0]
   Predicted  Actual
0          1       1
1          0       1
2          0       0
3          1       0
4          0       0


**Which approach(es) did you find helpful to improve your model performance?**

In [12]:
# your answer here

# Use Dropout to prevent overfitting, normalize the data
# Implement various hidden layers